Congrats! You just graduated UVA's BSDS program and got a job working at a movie studio in Hollywood. 

Your boss is the head of the studio and wants to know if they can gain a competitive advantage by predicting new movies that might get high imdb scores (movie rating). 

You would like to be able to explain the model to mere mortals but need a fairly robust and flexible approach so you've chosen to use decision trees to get started. 

In doing so, similar to  great data scientists of the past you remembered the excellent education provided to you at UVA in a undergrad data science course and have outline 20ish steps that will need to be undertaken to complete this task. As always, you will need to make sure to #comment your work heavily. 

 Footnotes: 
-	You can add or combine steps if needed
-	Also, remember to try several methods during evaluation and always be mindful of how the model will be used in practice.
- Make sure all your variables are the correct type (factor, character,numeric, etc.)

In [ ]:
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
import matplotlib.pyplot as plt
import graphviz

from sklearn.model_selection import train_test_split,GridSearchCV,RepeatedStratifiedKFold
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_graphviz 

In [ ]:
#1. Load the data
#Sometimes need to set the working directory back out of a folder that we create a file in

#import os
#os.listdir()
#print(os.getcwd())
#os.chdir('c:\\Users\\Brian Wright\\Documents\\3001Python\\DS-3001')

movie_metadata=pd.read_csv("../data/movie_metadata.csv")
movie_metadata.head()

2 Ensure all the variables are classified correctly including the target variable and collapse factor variables as needed.

In [ ]:
movie_metadata.info()

Any variable such as director name, actor name, or other strings will not be useful in making our decision tree.

In [ ]:
# Keeping the variables that will be impactful on IMDB rating(as well as target), dropping all others
movies_data = movie_metadata[['color', 'num_critic_for_reviews', 'duration', 'gross', 'num_voted_users','country', 'language', 'content_rating', 'budget', 'title_year', 'aspect_ratio', 'imdb_score']]
movies_data.head()

In [ ]:
movies_data.info()

In [ ]:
# We can make the variables color, country, language, and content rating into categorical variables
# For language and country we will make it into American/English vs. other(I'm biased)
print("Color levels:", movies_data['color'].unique())
print("Country levels:", movies_data['country'].unique())
print("Language levels:", movies_data['language'].unique())
print("Rating levels:", movies_data['content_rating'].unique())

In [ ]:
# For color our categories will be color vs. black and white. We do not need to collapse levels
movies_data['color'] = movies_data['color'].astype('category')

In [ ]:
for i in movies_data.index:
    if movies_data.loc[i, 'country'] != 'USA':
       movies_data.loc[i, 'country'] = 'Other'
movies_data['country'] = movies_data['country'].astype('category')


In [ ]:
for i in movies_data.index:
    if movies_data.loc[i, 'language'] != 'English':
       movies_data.loc[i, 'language'] = 'Other'
movies_data['language'] = movies_data['language'].astype('category')

In [ ]:
for i in movies_data.index:
    if movies_data.loc[i, 'content_rating'] not in ['G', 'PG', 'PG-13', 'R']:
       movies_data.loc[i, 'content_rating'] = 'Other'
movies_data['content_rating'] = movies_data['content_rating'].astype('category')

We need to encode our categorical variables as numeric

In [ ]:
movies_data[["color"]] = OrdinalEncoder().fit_transform(movies_data[["color"]])
print(movies_data["color"].value_counts()) 

In [ ]:
movies_data[["language"]] = OrdinalEncoder().fit_transform(movies_data[["language"]])
print(movies_data["language"].value_counts()) 

In [ ]:
movies_data[["country"]] = OrdinalEncoder().fit_transform(movies_data[["country"]])
print(movies_data["country"].value_counts()) 

In [ ]:
movies_data[["content_rating"]] = OrdinalEncoder().fit_transform(movies_data[["content_rating"]])
print(movies_data["content_rating"].value_counts()) 

In [ ]:
movies_data.info()

In [ ]:
movies_data.head()

3 Check for missing variables and correct as needed. Once you've completed the cleaning again create a function that will do this for you in the future. In the submission, include only the function and the function call.

In [ ]:
print(movies_data.isna().sum()) 

In [ ]:
# We're missing a fair amount of data for financial information. However, it is important that we have this information so we
# will drop any missing values
movies_data = movies_data.dropna()
movies_data.info()

We still have almost 4000 observations which will be sufficent.

4 Guess what, you don't need to scale the data, because DTs don't require this to be done, they make local greedy decisions...keeps getting easier, go to the next step.

5 Determine the baserate or prevalence for the classifier, what does this number mean?

In the context of 'high' IMDB scores(what we are trying to find), something about 7/10 would be considered high. Let's check what percentage of IMDB scores are > 7:

In [ ]:
# Encoding IMDB score as 1 if >7, 0 if not
movies_data['imdb_score'] = movies_data['imdb_score'].apply(lambda x: 1 if x > 7 else 0)

# Now you can calculate the proportion of high scores
high_score_ratio = movies_data['imdb_score'].mean()
print(high_score_ratio)


About 30% of our movies seems to be a good level of prevalence. We shall proceed.

6 Split your data into test, tune, and train. (80/10/10)

In [ ]:
# Splitting into our independent and dependent variables
X= movies_data.drop(columns='imdb_score')
y= movies_data.imdb_score

In [ ]:
# Train/tune/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, stratify= y, random_state=13)
X_tune, X_test, y_tune, y_test = train_test_split(X_test,y_test,  train_size = 0.50,stratify= y_test, random_state=72)

7 Create the kfold object for cross validation.

In [ ]:
kf = RepeatedStratifiedKFold(n_splits=10,n_repeats =5, random_state=42)

8 Create the scoring metric you will use to evaluate your model and the max depth hyperparameter (grid search) 

In [ ]:
# Using roc_auc, f1_score(good for skew), and r2_score(correlation)
scoring = ['roc_auc','f1','r2']

9 Build the classifier object 

In [ ]:
param={"max_depth" : [1,2,3,4,5,6,7,8,9,10,11]}
cl= DecisionTreeClassifier(random_state=500)
search = GridSearchCV(cl, param, scoring=scoring, n_jobs=-1, cv=kf,refit='roc_auc')

10 Use the kfold object and the scoring metric to find the best hyperparameter value for max depth via the grid search method.

A depth of 6 is best

11 Fit the model to the training data.

In [ ]:
# Fitting to the training data
model = search.fit(X_train, y_train)

12 What is the best depth value?

In [ ]:
best = model.best_estimator_
print(best)

The best depth is 6

13 Print out the model

14 View the results, comment on how the model performed using the metrics you selected.

In [ ]:
#Scores: 
auc = model.cv_results_['mean_test_roc_auc']
f1 = model.cv_results_['mean_test_f1']
r2 = model.cv_results_['mean_test_r2']

SD_auc = model.cv_results_['std_test_roc_auc']
SD_f1 = model.cv_results_['std_test_f1']
SD_r2= model.cv_results_['std_test_r2']

#Parameter:
depth= np.unique(model.cv_results_['param_max_depth']).data

#Build DataFrame:
final_model = pd.DataFrame(list(zip(depth, auc, f1, r2,SD_auc,SD_f1,SD_r2)),
               columns =['depth','auc','f1','r2','aucSD','f1SD','r2SD'])

#Let's take a look
final_model.style.hide(axis='index')


We can see here that auc is highest at a depth of 6. f1 score also seems to perform best around 6

15 Which variables appear to be contributing the most (variable importance) 

In [ ]:
varimp=pd.DataFrame(best.feature_importances_,index = X.columns,columns=['importance']).sort_values('importance', ascending=False)
print(varimp)

The most important variables are the number of users who voted, budget, and language(english vs. other)

16 Use the predict method on the tune data and print out the results.

In [ ]:
from sklearn.metrics import accuracy_score
 
y_pred = model.predict(X_tune)
accuracy = accuracy_score(y_tune, y_pred)
print("Accuracy:", accuracy)

The model performs decently well on the tune data, with an accuracy of about 80%. 

In [ ]:
print(ConfusionMatrixDisplay.from_estimator(best,X_tune,y_tune, display_labels = ['ave/poor','strong'], colorbar=False))

The confusion matrix tells us that the model does well at predicting a movie will be average/poor when it is but is not as good at classifying strong movies as strong.

17 How does the model perform on the tune data?

18 Print out the confusion matrix for the tune data, what does it tell you about the model?

19 What are the top 3 movies based on the tune set? Which variables are most important in predicting the top 3 movies?

In [ ]:
# Predict probabilities on the tune set
y_probs = best.predict_proba(X_tune)[:, 1]

# Attach probabilities to the original tune set
tune_with_probs = X_tune.copy()
tune_with_probs['predicted_prob'] = y_probs

top_3 = tune_with_probs.sort_values(by='predicted_prob', ascending=False).head(3)
print(top_3)


The movies with the highest likelihood of being highly rated on IMDB were "The Matrix: Reloaded", "Casino Royale", and "Die Hard". The number of reviews, budget, and duration were most important.

20 Use a different hyperparameter for the grid search function and go through the process above again using the tune set. 

In [ ]:
param_1={"min_samples_split":[5,10,15,20,25],}
search_2 = GridSearchCV(cl, param_1, scoring=scoring, n_jobs=-1, cv=kf,refit='roc_auc')

model_2 = search_2.fit(X_train, y_train)
#Scores: 
auc_2 = model_2.cv_results_['mean_test_roc_auc']
f1_2 = model_2.cv_results_['mean_test_f1']
r2_2 = model_2.cv_results_['mean_test_r2']

SD_auc_2 = model_2.cv_results_['std_test_roc_auc']
SD_f1_2 = model_2.cv_results_['std_test_f1']
SD_r2_2 = model_2.cv_results_['std_test_r2']

#Parameter:
min_samples = model_2.cv_results_['param_min_samples_split'].data

#Build DataFrame:
final_model = pd.DataFrame(list(zip(min_samples, auc_2, f1_2, r2_2,SD_auc_2,SD_f1_2,SD_r2_2)),
               columns =['depth','auc','f1','r2','aucSD','f1SD','r2SD'])

#Let's take a look
final_model.style.hide(axis='index')


21 Did the model improve with the new hyperparameter search?

The other model performed slightly better based on auc.

22 Using the better model, predict the test data and print out the results.

In [ ]:
y_pred = best.predict(X_test)               # predicted class (0 or 1)
y_proba = best.predict_proba(X_test)[:, 1]  # predicted probability of class 1

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))


23 Summarize what you learned along the way and make recommendations to your boss on how this could be used moving forward, being careful not to over promise.